# PolicyGuard-4B: 100 Test Cases Inference Pipeline

This notebook executes the minimal **PolicyGuard-4B** inference pipeline on 100 preformatted prompt test cases.

### Pipeline Purpose
1. Load pretrained `Rakancorle1/PolicyGuard-4B`.
2. Read 100 preformatted `.txt` prompt files from Google Drive.
3. Run greedy decoding (`do_sample=False`, `max_new_tokens=4`).
4. Decode and strictly parse responses into `violation`, `no_violation`, or `UNKNOWN`.
5. Stream predictions to `predictions.jsonl`, `predictions.csv`, and `summary.json` on Google Drive.

## Cell 1: Environment Setup

> **Important:** Ensure you have selected a GPU runtime (**Runtime -> Change runtime type -> T4 GPU or higher**).

In [ ]:
import torch

print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA is available! GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"Device Count: {torch.cuda.device_count()}")
else:
    print("WARNING: CUDA is NOT available. PolicyGuard inference requires a GPU runtime.")
    print("Please go to: Runtime -> Change runtime type -> Select T4 GPU or higher.")

# Install minimal required packages
!pip install -q transformers>=4.40.0 accelerate>=0.28.0 safetensors>=0.4.0


## Cell 2: Repository and Drive Setup

Mount Google Drive to access input prompts and persist experiment outputs across sessions.

In [ ]:
import os
import sys
from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive")

# Ensure repository directory is in sys.path to import policyguard_pipeline
repo_path = "/content/GuardSLM"
if os.path.exists(repo_path) and repo_path not in sys.path:
    sys.path.insert(0, repo_path)

if "." not in sys.path:
    sys.path.insert(0, ".")

from policyguard_pipeline import load_model, generate_response, parse_output, run_batch
print("Successfully imported policyguard_pipeline.")


## Cell 3: User Configuration

Configure input/output directory paths and model settings.
- `INPUT_DIR`: Google Drive folder containing your 100 preformatted `.txt` prompt files.
- `OUTPUT_DIR`: Destination folder on Google Drive for predictions and run summary.

In [ ]:
MODEL_ID = "Rakancorle1/PolicyGuard-4B"
INPUT_DIR = "/content/drive/MyDrive/PolicyGuardInputs"
OUTPUT_DIR = "/content/drive/MyDrive/PolicyGuardOutputs"
EXPECTED_CASES = 100
MAX_NEW_TOKENS = 4

print("Configuration Parameters:")
print(f"  Model ID:        {MODEL_ID}")
print(f"  Input Directory: {INPUT_DIR}")
print(f"  Output Directory:{OUTPUT_DIR}")
print(f"  Expected Cases:  {EXPECTED_CASES}")
print(f"  Max New Tokens:  {MAX_NEW_TOKENS}")


## Cell 4: Input Validation

Verify that `INPUT_DIR` exists and contains exactly 100 `.txt` prompt files.

In [ ]:
import re

def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r"(\d+)", s)]

if not os.path.isdir(INPUT_DIR):
    raise FileNotFoundError(
        f"Input directory {INPUT_DIR} not found. Please upload your 100 .txt prompt files "
        "to Google Drive and update INPUT_DIR."
    )

txt_files = sorted([f for f in os.listdir(INPUT_DIR) if f.endswith(".txt")], key=natural_sort_key)
print(f"Found {len(txt_files)} .txt files in {INPUT_DIR}.")

if len(txt_files) != EXPECTED_CASES:
    print(f"[NOTE] Found {len(txt_files)} files (configured expectation is {EXPECTED_CASES}).")
else:
    print(f"[VALIDATION PASSED] Exactly {EXPECTED_CASES} test case files detected.")

print("\nFirst 5 discovered files:")
for f in txt_files[:5]:
    print(f"  - {f}")
if len(txt_files) > 5:
    print(f"  ... ({len(txt_files) - 5} more files)")


## Cell 5: Load PolicyGuard-4B Model

Loads the pretrained tokenizer and model once into memory.

In [ ]:
tokenizer, model = load_model(model_id=MODEL_ID)
print("PolicyGuard-4B model and tokenizer loaded successfully.")


## Cell 6: Smoke Test (3 Cases)

Execute 3 cases into a separate test folder to verify the pipeline without touching full experiment outputs.

In [ ]:
smoke_test_dir = "/content/drive/MyDrive/PolicyGuardSmokeTest"

print("Running 3-case smoke test...")
smoke_summary = run_batch(
    input_dir=INPUT_DIR,
    output_dir=smoke_test_dir,
    model=model,
    tokenizer=tokenizer,
    limit=3,
    resume=False,
    model_id=MODEL_ID,
    max_new_tokens=MAX_NEW_TOKENS
)
print("Smoke test completed successfully!")


## Cell 7: Full Experiment (100 Cases)

Execute all 100 test cases sequentially with progress logging and streaming crash resilience.

In [ ]:
print(f"Executing Full Experiment on {INPUT_DIR}...")

full_summary = run_batch(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    model=model,
    tokenizer=tokenizer,
    limit=None,
    resume=True,
    model_id=MODEL_ID,
    max_new_tokens=MAX_NEW_TOKENS
)


## Cell 8: Output Collection & Inspection

Display run metrics and preview predictions CSV.

In [ ]:
import pandas as pd

print("=" * 60)
print("EXPERIMENT RUN SUMMARY")
print("=" * 60)
print(f"Total Inputs Evaluated:    {full_summary[number_of_inputs]}")
print(f"Successfully Generated:    {full_summary[successfully_generated_count]}")
print(f"Violations:                {full_summary[count_violation]}")
print(f"No Violations:             {full_summary[count_no_violation]}")
print(f"UNKNOWN Responses:         {full_summary[count_UNKNOWN]}")
print(f"Average Latency:           {full_summary[average_latency_ms]} ms")
print(f"Median Latency:            {full_summary[median_latency_ms]} ms")
print(f"Total Execution Time:      {full_summary[total_runtime_seconds]} s")
print("\nSaved Output Files on Google Drive:")
print(f"  JSONL:    {full_summary[predictions_jsonl_path]}")
print(f"  CSV:      {full_summary[predictions_csv_path]}")
print(f"  Summary:  {full_summary[summary_json_path]}")

csv_path = full_summary[predictions_csv_path]
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print("\nPredictions CSV Preview (First 10 rows):")
    display(df.head(10))


## Cell 9: Cleanup

Safely release model memory and empty GPU cache.

In [ ]:
import gc

print("Releasing model and clearing GPU memory...")
del model
del tokenizer
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU cache cleared successfully.")
print("Cleanup complete.")
